# Pareto multi-criteria routing lab
Compare route alternatives when travel time and distance (or other additive costs) cannot be reduced to one universally correct score.


In [ ]:
from pathlib import Path
import sys
for parent in (Path.cwd(), *Path.cwd().parents):
    package = parent / "implementations" / "python" / "shortest-paths"
    if package.is_dir():
        sys.path.insert(0, str(package))
        break
else:
    raise RuntimeError("Could not locate implementations/python/shortest-paths")
from pareto_paths import pareto_shortest_paths


## 1. Define a network with trade-offs
Each edge carries a two-element vector: `(time, distance)`. Three routes make different trade-offs; a direct route is deliberately dominated.


In [ ]:
graph = {
    "s": [("a", (1, 8)), ("b", (7, 1)), ("c", (4, 4)), ("t", (12, 12))],
    "a": [("t", (1, 2))],
    "b": [("t", (3, 1))],
    "c": [("t", (4, 4))],
    "t": [],
}
frontier = pareto_shortest_paths(graph, "s", "t", n_objectives=2)
for route in frontier:
    print(route)


## 2. Inspect dominance
A label is dominated when another label is no worse in every objective and strictly better in at least one. The direct route `(12, 12)` is therefore excluded.


In [ ]:
costs = [route["costs"] for route in frontier]
assert costs == [(2.0, 10.0), (8.0, 8.0), (10.0, 2.0)]
print("Nondominated cost vectors:", costs)


## 3. Explore a preference after computing the frontier
The algorithm returns the frontier without choosing a preferred route. A decision-maker can apply a preference only after seeing the trade-offs.


In [ ]:
for time_weight, distance_weight in ((1, 0), (0, 1), (0.5, 0.5)):
    chosen = min(frontier, key=lambda r: time_weight*r["costs"][0] + distance_weight*r["costs"][1])
    print((time_weight, distance_weight), chosen)


## Interpretation and limits
The label-setting method is exact for finite graphs with nonnegative additive edge-cost vectors, but the Pareto frontier can grow exponentially. Equal cost vectors retain one representative path. Negative costs, automatic preference selection, turn-expanded state, time-dependent routing and timetable integration are outside this baseline.
